# Supplementary Figure 13 - TNBC spatial biomarker statistics

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `Ali_TNBC_2023/NTPublic/sfplot/sfplot_TNBC.ipynb` and local/A100 mirrors.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
library(data.table)
library(dplyr)
library(tidyr)
library(ggplot2)

PROJECT_DIR <- "Y:/long/publication_datasets/Ali_TNBC_2023/NTPublic"
setwd(PROJECT_DIR)
source(here::here("code/header.R"))
source("Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis/Rcode/sfplotR/R/compute_cophenetic_distances_from_df.R")
source("Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis/Rcode/sfplotR/R/plot_cophenetic_heatmap.R")

cells <- as.data.frame(getCells(curated = TRUE, allCells = TRUE))
clinical <- read.csv(file.path(PROJECT_DIR, "data/derived/clinical.csv"))


In [ ]:
OUTPUT_DIR <- "outputs/supp_fig_13_TNBC_spatial_biomarkers"
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Compute marker-pair SSS per patient and biopsy phase.
markers <- unique(cells$Label)
combinations <- t(combn(markers, 2))
all_results <- list()
for (phase in unique(cells$BiopsyPhase)) {
  for (patient in unique(cells$PatientID)) {
    submeta <- cells %>% filter(PatientID == patient, BiopsyPhase == phase)
    if (nrow(submeta) < 50) next
    result <- compute_cophenetic_distances_from_df(submeta, cluster_col = "Label", x_col = "Location_Center_X", y_col = "Location_Center_Y", method = "average")
    row_coph <- result$row_cophenetic_df
    rows <- lapply(seq_len(nrow(combinations)), function(i) {
      m1 <- combinations[i, 1]; m2 <- combinations[i, 2]
      value <- if (m1 %in% rownames(row_coph) && m2 %in% colnames(row_coph)) row_coph[m1, m2] else NA_real_
      data.frame(PatientID = patient, BiopsyPhase = phase, Marker_1 = m1, Marker_2 = m2, result = value)
    })
    all_results[[paste(patient, phase, sep = "_")]] <- bind_rows(rows)
  }
}
final_results <- bind_rows(all_results)
final_results <- merge(final_results, clinical[, c("PatientID", "BiopsyPhase", "pCR", "Arm")], by = c("PatientID", "BiopsyPhase"), all.x = TRUE)
write.csv(final_results, file.path(OUTPUT_DIR, "TNBC_marker_pair_SSS.csv"), row.names = FALSE)


In [ ]:
wilcox_results <- final_results %>%
  filter(!is.na(result), !is.na(pCR)) %>%
  group_by(BiopsyPhase, Arm, Marker_1, Marker_2) %>%
  summarise(n_pCR = sum(pCR == "pCR"), n_RD = sum(pCR == "RD"), p_value = if (n_pCR >= 2 && n_RD >= 2) wilcox.test(result[pCR == "pCR"], result[pCR == "RD"], exact = FALSE)$p.value else NA_real_, .groups = "drop")
write.csv(wilcox_results, file.path(OUTPUT_DIR, "wilcox_results_TNBC.csv"), row.names = FALSE)
